# Table 3: all twelve accessory families

Companion to **Recurrence formulas for Taylor expansions at CM points on genus-one and genus-two Shimura curves**.
Choose the **SageMath** kernel and **Restart Kernel and Run All**. This notebook is self-contained and uses exact arithmetic.

Starting with the models $y^2=P(x)$ and differentials $\omega=h(x)dx/y$, we derive
$H(u(x))=u_x(x)^2P(x)/h(x)^2$, the prescribed principal parts of $\mathcal R$, and the free accessory basis.
The models, eigendifferentials and complete elliptic-fiber list are the geometric inputs of Tables 1 and 3 and Appendix B;
their identification with Shimura curves uses the cited geometric results. No numerical CM coefficients are assumed here.

In [1]:
Rx.<x> = PolynomialRing(QQ)
K = Rx.fraction_field()
Ru.<u> = PolynomialRing(QQ)
U = Ru.fraction_field()
S.<X> = PolynomialRing(U)

# Table 1: y^2 = model[N]. For genus two, h=x for a and h=1 for b.
model = {
    14: -x^4+13*x^2-128,      15: -3*x^4-82*x^2-27,
    21: -7*x^4+94*x^2-343,    33: -3*x^4-10*x^2-243,
    34: -3*x^4+26*x^3-53*x^2-26*x-3,
    46: -x^4+45*x^2-512,      26: -2*x^6+19*x^4-24*x^2-169,
    38: -16*x^6-59*x^4-82*x^2-19,
    58: -2*x^6-78*x^4-862*x^2-1682}

# Each tuple is (u-value, elliptic order e, quotient ramification r, ord(omega) m).
fibers = {
    '14a': [(oo,2,2,0)],          '15a': [(oo,3,2,0)],
    '21a': [(49,2,1,0)],          '33a': [(0,3,2,0),(1,2,1,0)],
    '34a': [(16/15,3,1,0)],       '46a': [(oo,2,2,0),(64,3,1,0)],
    '26a': [(0,1,2,1)],           '26b': [(0,1,2,1)],
    '38a': [(0,1,2,1),(oo,2,2,0)], '38b': [(0,2,2,1)],
    '58a': [(0,1,2,1),(-27,3,1,0)],
    '58b': [(0,1,2,1),(-1/27,3,1,0)]}

## Local coefficient and quotient calculation

For $z=v^{1/e}$ and $t=\alpha v^{m+1}/(m+1)$, the Schwarzian chain rule gives
$$\{z,t\}=\frac{(m+1)^2-e^{-2}}{2\alpha^2}v^{-2m-2}.$$
We verify all four local types used below. Higher analytic terms do not change this leading coefficient.
To derive $H$ we reduce $u_x^2P/h^2$ modulo the quadratic relation between $x$ and $u$;
the remainder must be constant in $x$. We also check smoothness, quotient degree and the involution on the differential.

In [2]:
v, alpha = SR.var('v alpha')
def schwarzian(f):
    return diff(f,v,3)/diff(f,v) - 3/2*(diff(f,v,2)/diff(f,v))^2
for m,e in [(0,2),(0,3),(1,1),(1,2)]:
    z, t = v^(1/e), alpha*v^(m+1)/(m+1)
    actual = (schwarzian(z)-schwarzian(t))/diff(t,v)^2
    expected = ((m+1)^2-1/e^2)/(2*alpha^2)*v^(-2*m-2)
    assert (actual-expected).simplify_full() == 0

def model_data(form):
    N = ZZ(form[:-1]); P = model[N]; genus = (P.degree()-2)//2
    h = K(x if genus==2 and form.endswith('a') else 1)
    coord = K(x-1/x if N==34 else x^(-2) if form.endswith('b') else x^2)
    relation = X^2-u*X-1 if N==34 else u*X^2-1 if form.endswith('b') else X^2-u
    pulled_H = coord.derivative()^2*P/h^2
    remainder = (S(pulled_H.numerator()) * S(pulled_H.denominator()).inverse_mod(relation)) % relation
    assert remainder.degree() == 0
    H = Ru(remainder[0])
    assert K(H(coord)) == pulled_H and P.gcd(P.derivative()) == 1
    sigma, yfactor = (K(-1/x), K(-1/x^2)) if N==34 else (K(-x), K(1))
    assert P(sigma) == yfactor^2*P and coord(sigma) == coord
    assert h(sigma)*sigma.derivative()/(yfactor*h) in [1,-1]
    assert max(coord.numerator().degree(),coord.denominator().degree()) == 2
    return P,h,coord,H,genus

## Descended principal parts and free parameters

Write $\kappa_e=(1-e^{-2})/2$. The prescribed terms are
$\kappa_e H(a)/(u-a)^2$ at an unramified elliptic fiber,
$\kappa_e H'(a)/(4(u-a))$ at a ramified elliptic fiber,
$\kappa_{2e}H(a)/(u-a)^2$ at a simple differential-zero fiber, and
$\kappa_e\operatorname{lc}(P)u$ at the listed infinite fibers.
The code checks the corresponding local behavior from each model.

After subtracting these terms, the free part is
$L_{\mathbf P^1}(\Delta)$ with
$\Delta=\sum_a\lfloor(2m_a+\mathbf1_{e_a>1})/r_a\rfloor[a]$.
Here each coefficient is $0$ or $1$, with no contribution at infinity:
the basis is $1$ and $1/(u-a)$ for each point in its support.
We check linear independence, so the dimension is $d=1+\deg\Delta$.

In [3]:
def kappa(e): return (1-QQ(1)/e^2)/2

def derive_family(form):
    P,h,coord,H,genus = model_data(form)
    terms, support = [], []
    for a,e,r,m in fibers[form]:
        delta = (2*m+ZZ(e>1))//r
        assert delta in [0,1]
        if delta:
            assert a != oo
            support.append(a)
        if a == oo:
            assert (m,r)==(0,2) and coord(1/x)==x^(-2)
            dvdt2 = K(x^4*P(1/x)/h(1/x)^2)
            assert dvdt2(0) == P.leading_coefficient() != 0
            terms.append(U(kappa(e)*dvdt2(0)*u))
        elif (m,r)==(0,1):
            assert H(a) != 0
            terms.append(U(kappa(e)*H(a)/(u-a)^2))
        elif (m,r)==(0,2):
            assert a==0 and coord==x^2 and P(0)!=0
            assert H(a)==0 and H.derivative()(a)!=0
            terms.append(U(kappa(e)*H.derivative()(a)/(4*(u-a))))
        else:
            assert (m,r)==(1,2) and a==0 and genus==2
            # At the zero use v=x for a, v=1/x for b (x is the local variable here).
            local_x = x if form.endswith('a') else 1/x
            omega2 = K(h(local_x)^2*local_x.derivative()^2/P(local_x))
            alpha2 = K(omega2/x^2)
            assert coord(local_x)==x^2 and alpha2(0)!=0 and alpha2.derivative()(0)==0
            assert H(0)==4/alpha2(0)
            terms.append(U(kappa(2*e)*H(0)/u^2))
    basis = [U(1)] + [U(1/(u-a)) for a in support]
    denominator = prod(u-a for a in support)
    vectors = [Ru(b*denominator).padded_list(len(basis)) for b in basis]
    assert matrix(QQ,vectors).rank() == len(basis)
    return H, sum(terms,U(0)), basis, terms

## Compare with every row of Table 3

The expected fixed parts and accessory bases below are separate transcriptions of the printed table.
They are used only after deriving the families. Equality is in $\mathbf Q(u)$; equality of the bases also checks
the allowed poles and all accessory parameters. The displayed family is the derived expression.

In [4]:
# form : (printed fixed part, printed nonconstant accessory poles, printed dimension)
printed = {
    '14a': (-3*u/8, [], 1),
    '15a': (-4*u/3, [], 1),
    '21a': (-921984/(u-49)^2, [49], 2),
    '33a': (-384/(u-1)^2-108/u, [1], 2),
    '34a': (-1336336/(16875*(u-16/15)^2), [16/15], 2),
    '46a': (-196608/(u-64)^2-3*u/8, [64], 2),
    '26a': (-507/(2*u^2), [0], 2),
    '26b': (-3/u^2, [0], 2),
    '38a': (-57/(2*u^2)-6*u, [0], 2),
    '38b': (-30/u^2, [0], 2),
    '58a': (-2523/u^2+65536/(9*(u+27)^2), [0,-27], 3),
    '58b': (-3/u^2-65536/(177147*(u+1/27)^2), [0,-1/27], 3)}
assert set(printed) == set(fibers) and len(printed)==12
from IPython.display import display, Math
def term_latex(f):
    scale = lcm([c.denominator() for c in f.numerator().list()+f.denominator().list()])
    n,d = scale*f.numerator(), scale*f.denominator()
    return latex(n) if d==1 else r'\frac{'+latex(n)+r'}{'+latex(d.factor())+'}'
rows = []
for form in fibers:
    H,fixed,basis,terms = derive_family(form)
    target,poles,d = printed[form]
    assert fixed==U(target) and basis==[U(1)]+[U(1/(u-a)) for a in poles]
    assert len(basis)==d
    parameters = ['B'] if d==2 else ['B_0','B_1'] if d==3 else []
    expression = 'C' + ''.join(' + '+b+' \\cdot '+term_latex(f) for b,f in zip(parameters,basis[1:]))
    expression += ''.join(' + '+term_latex(f) if f.numerator().leading_coefficient()>0 else ' - '+term_latex(-f) for f in terms)
    rows.append(r'\mathrm{'+form+'}&'+expression+'&'+str(d))
display(Math(r'\begin{array}{c|l|c}f&\mathcal R(u)&d\\\hline'+r'\\'.join(rows)+r'\end{array}'))
print('PASS: all 12 Table 3 families, local leading coefficients, and accessory dimensions.')

<IPython.core.display.Math object>

PASS: all 12 Table 3 families, local leading coefficients, and accessory dimensions.
